Agents = LLMs (with tools, retrievers, or roles).

Nodes = Functions that process input → output.

Edges = Define how output moves between nodes.

State = Carries conversation + memory across nodes.

So a multi-agent system = graph of agents.

In [2]:
from typing import Dict, Any, Optional
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field

# Define a proper State class
class ConversationState(BaseModel):
    agent1_msg: Optional[str] = None
    agent2_msg: Optional[str] = None
    messages: list = Field(default_factory=list)

# Define LLMs
assistant1 = ChatOpenAI(model="gpt-4o-mini", temperature=0)
assistant2 = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Node 1
def agent1_node(state: ConversationState):
    response = assistant1.invoke("You are Agent 1. Say hello to Agent 2.")
    state.agent1_msg = response.content
    state.messages.append(f"Agent 1: {response.content}")
    return state

# Node 2
def agent2_node(state: ConversationState):
    if state.agent1_msg:
        response = assistant2.invoke(f"You are Agent 2. Reply to Agent 1: {state.agent1_msg}")
        state.agent2_msg = response.content
        state.messages.append(f"Agent 2: {response.content}")
    return state

# Build Graph
workflow = StateGraph(ConversationState)
workflow.add_node("agent1", agent1_node)
workflow.add_node("agent2", agent2_node)

workflow.set_entry_point("agent1")
workflow.add_edge("agent1", "agent2")
workflow.add_edge("agent2", END)

app = workflow.compile()

# Run with initial state
initial_state = ConversationState()
final_state = app.invoke(initial_state)
print("Final State:", final_state)

Final State: {'agent1_msg': 'Hello, Agent 2! How are you today?', 'agent2_msg': "Hello, Agent 1! I'm doing well, thank you! How about you? What’s on the agenda today?", 'messages': ['Agent 1: Hello, Agent 2! How are you today?', "Agent 2: Hello, Agent 1! I'm doing well, thank you! How about you? What’s on the agenda today?"]}


In [6]:
from typing import Dict, Any, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Use simple dictionary state - much easier with LangGraph!
assistant1 = ChatOpenAI(model="gpt-4o-mini", temperature=0)
assistant2 = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def agent1_node(state: Dict[str, Any]):
    """Agent 1 initiates conversation"""
    response = assistant1.invoke("You are Agent 1. Introduce yourself to Agent 2.")
    
    # Initialize lists if they don't exist
    if "conversation_history" not in state:
        state["conversation_history"] = []
    
    state["agent1_message"] = response.content
    state["conversation_history"].append(f"🤖 Agent 1: {response.content}")
    return state

def agent2_node(state: Dict[str, Any]):
    """Agent 2 responds to Agent 1"""
    if "agent1_message" in state and state["agent1_message"]:
        prompt = f"""You are Agent 2. Respond to Agent 1's message:
        {state['agent1_message']}
        
        Be friendly and engaging in your response."""
        
        response = assistant2.invoke(prompt)
        state["agent2_message"] = response.content
        state["conversation_history"].append(f"🤖 Agent 2: {response.content}")
    
    return state

# Build the workflow with dict state
workflow = StateGraph(dict)  # Use dict instead of Pydantic model
workflow.add_node("agent1", agent1_node)
workflow.add_node("agent2", agent2_node)

workflow.set_entry_point("agent1")
workflow.add_edge("agent1", "agent2")
workflow.add_edge("agent2", END)

app = workflow.compile()

# Execute with empty dict
result = app.invoke({})

print("🤖 Conversation Complete!")
print("=" * 50)
for message in result.get("conversation_history", []):
    print(message)
print("=" * 50)
print("Final state:", result)

🤖 Conversation Complete!
🤖 Agent 1: Hello, Agent 2. I’m Agent 1, and I’m here to collaborate and assist in any way I can. Whether it’s sharing information, strategizing, or tackling challenges together, I’m ready to get started. How can I help you today?
🤖 Agent 2: Hello, Agent 1! It’s great to connect with you. I appreciate your willingness to collaborate! I think we could start by discussing our current objectives and any challenges we might be facing. If you have any specific areas where you think we could strategize together, I’d love to hear your thoughts. Let’s make this a productive partnership! What’s on your mind?
Final state: {'conversation_history': ['🤖 Agent 1: Hello, Agent 2. I’m Agent 1, and I’m here to collaborate and assist in any way I can. Whether it’s sharing information, strategizing, or tackling challenges together, I’m ready to get started. How can I help you today?', '🤖 Agent 2: Hello, Agent 1! It’s great to connect with you. I appreciate your willingness to coll

### More Realistic: Researcher + Writer Agents

In [10]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Researcher agent
def researcher(state: dict):
    topic = state.get("topic", "AI and machine learning")
    print(f"🔍 Researching topic: {topic}")
    
    response = llm.invoke(f"You are a researcher. Gather 3 key facts about {topic}.")
    state["research"] = response.content
    print("✅ Research completed")
    return state

# Writer agent
def writer(state: dict):
    research = state.get("research", "")
    topic = state.get("topic", "the topic")
    
    print(f"✍️ Writing article about: {topic}")
    response = llm.invoke(f"""You are a writer. Write a short blog post using this research:

RESEARCH:
{research}

Write in a friendly, informative tone.""")
    
    state["draft"] = response.content
    print("✅ Draft completed")
    return state

# Graph setup - Use dict instead of ArticleState class
workflow = StateGraph(dict)
workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", END)

app = workflow.compile()

# Run it with proper initial state
print("🚀 Starting article generation...")
initial_state = {"topic": "LangChain multi-agent systems"}
final_state = app.invoke(initial_state)

print("\n" + "="*60)
print("📝 FINAL DRAFT:")
print("="*60)
if final_state and "draft" in final_state:
    print(final_state["draft"])
else:
    print("❌ Error: No draft was generated")
    print("Final state:", final_state)

print("\n" + "="*60)
print("🔍 RESEARCH NOTES:")
print("="*60)
if final_state and "research" in final_state:
    print(final_state["research"])

🚀 Starting article generation...
🔍 Researching topic: LangChain multi-agent systems
✅ Research completed
✍️ Writing article about: LangChain multi-agent systems
✅ Draft completed

📝 FINAL DRAFT:
**Unlocking the Power of LangChain Multi-Agent Systems**

In the ever-evolving landscape of artificial intelligence, LangChain is making waves with its innovative multi-agent systems. If you’re curious about how these systems work and what makes them so special, you’re in the right place! Let’s dive into three key features that set LangChain apart.

**1. Modular Architecture: Customization at Your Fingertips**

One of the standout features of LangChain’s multi-agent systems is their modular architecture. Think of it as a toolkit where developers can mix and match various components—like language models, tools, and memory—to create agents that fit their specific needs. This flexibility means you can design complex workflows and interactions tailored to your unique use case, whether it’s for a ch

In [11]:
from typing import Dict, Any, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Use simple dictionary state - much easier with LangGraph!
assistant1 = ChatOpenAI(model="gpt-4o-mini", temperature=0)
assistant2 = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def agent1_node(state: Dict[str, Any]):
    """Agent 1 initiates conversation"""
    response = assistant1.invoke("You are Agent 1. Introduce yourself to Agent 2.")
    
    # Initialize lists if they don't exist
    if "conversation_history" not in state:
        state["conversation_history"] = []
    
    state["agent1_message"] = response.content
    state["conversation_history"].append(f"🤖 Agent 1: {response.content}")
    return state

def agent2_node(state: Dict[str, Any]):
    """Agent 2 responds to Agent 1"""
    if "agent1_message" in state and state["agent1_message"]:
        prompt = f"""You are Agent 2. Respond to Agent 1's message:
        {state['agent1_message']}
        
        Be friendly and engaging in your response."""
        
        response = assistant2.invoke(prompt)
        state["agent2_message"] = response.content
        state["conversation_history"].append(f"🤖 Agent 2: {response.content}")
    
    return state

# Build the workflow with dict state
workflow = StateGraph(dict)  # Use dict instead of Pydantic model
workflow.add_node("agent1", agent1_node)
workflow.add_node("agent2", agent2_node)

workflow.set_entry_point("agent1")
workflow.add_edge("agent1", "agent2")
workflow.add_edge("agent2", END)

app = workflow.compile()

# Execute with empty dict
result = app.invoke({})

print("🤖 Conversation Complete!")
print("=" * 50)
for message in result.get("conversation_history", []):
    print(message)
print("=" * 50)
print("Final state:", result)

🤖 Conversation Complete!
🤖 Agent 1: Hello, Agent 2. I’m Agent 1, and I’m here to collaborate and share insights. Together, we can tackle any challenges that come our way. How can I assist you today?
🤖 Agent 2: Hello, Agent 1! It’s great to connect with you. I’m excited about the opportunity to collaborate and share insights. I could use your expertise on a few projects I’m currently working on. Let’s brainstorm some ideas together! What’s on your mind today?
Final state: {'conversation_history': ['🤖 Agent 1: Hello, Agent 2. I’m Agent 1, and I’m here to collaborate and share insights. Together, we can tackle any challenges that come our way. How can I assist you today?', '🤖 Agent 2: Hello, Agent 1! It’s great to connect with you. I’m excited about the opportunity to collaborate and share insights. I could use your expertise on a few projects I’m currently working on. Let’s brainstorm some ideas together! What’s on your mind today?'], 'agent1_message': 'Hello, Agent 2. I’m Agent 1, and I

## robust

In [16]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Use simple dict instead of custom class
# ArticleState class is causing the issue - LangGraph works better with plain dict

# Researcher agent
def researcher(state: dict):
    topic = state.get("topic")
    if not topic:
        topic = "AI and machine learning"
        state["topic"] = topic
    
    print(f"🔍 Researching: {topic}")
    response = llm.invoke(f"""You are a research assistant. Gather 3-5 key insights about {topic}.
Focus on practical applications and recent developments.""")
    
    state["research"] = response.content
    print("✅ Research completed")
    return state

# Writer agent
def writer(state: dict):
    research = state.get("research", "")
    topic = state.get("topic", "the topic")
    
    if not research:
        research_response = llm.invoke(f"Provide 3 key points about {topic}")
        research = research_response.content
        state["research"] = research
    
    print(f"✍️ Writing article about: {topic}")
    response = llm.invoke(f"""You are a technical writer. Create a engaging blog post about {topic}.

Research Notes:
{research}

Write in a friendly, informative tone with clear examples.""")
    
    state["draft"] = response.content
    print("✅ Draft completed")
    return state

# Graph setup - USE dict INSTEAD OF ArticleState
workflow = StateGraph(dict)
workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", END)

app = workflow.compile()

# Run it - FIXED: Use regular dictionaries
print("🤖 Generating article about LangChain...")
state1 = app.invoke({"topic": "LangChain multi-agent systems"})
print("📝 Draft:", state1["draft"][:200] + "...")

print("\n" + "="*50)
print("🤖 Generating article about Quantum Computing...")
state2 = app.invoke({"topic": "Quantum Computing"})
print("📝 Draft:", state2["draft"][:200] + "...")

print("\n" + "="*50)
print("🤖 Generating article with default topic...")
state3 = app.invoke({})
print("📝 Draft:", state3["draft"][:200] + "...")

🤖 Generating article about LangChain...
🔍 Researching: LangChain multi-agent systems
✅ Research completed
✍️ Writing article about: LangChain multi-agent systems
✅ Draft completed
📝 Draft: # Unlocking the Power of Collaboration: Exploring LangChain Multi-Agent Systems

In the ever-evolving landscape of artificial intelligence, the ability to harness the power of language models has open...

🤖 Generating article about Quantum Computing...
🔍 Researching: Quantum Computing
✅ Research completed
✍️ Writing article about: Quantum Computing
✅ Draft completed
📝 Draft: # Unlocking the Future: The Exciting World of Quantum Computing

Welcome to the fascinating realm of quantum computing! If you’ve ever wondered how this cutting-edge technology could reshape our world...

🤖 Generating article with default topic...
🔍 Researching: AI and machine learning
✅ Research completed
✍️ Writing article about: AI and machine learning
✅ Draft completed
📝 Draft: # The Transformative Power of AI and Machine Lea

### simpler version


In [14]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def researcher(state):
    topic = state.get("topic", "AI technology")
    print(f"Researching: {topic}")
    
    response = llm.invoke(f"List 3 main points about {topic} for a blog article.")
    return {"research": response.content, "topic": topic}

def writer(state):
    research = state.get("research", "")
    response = llm.invoke(f"Write a short blog post based on: {research}")
    return {"draft": response.content}

# Build graph
workflow = StateGraph(dict)
workflow.add_node("research", researcher)
workflow.add_node("write", writer)

workflow.set_entry_point("research")
workflow.add_edge("research", "write")
workflow.add_edge("write", END)

app = workflow.compile()

# Test runs
print("🚀 Testing article generation system...\n")

test_cases = [
    {"topic": "LangChain multi-agent systems"},
    {"topic": "Quantum Computing"},
    {}  # Empty - should use default
]

for i, test_case in enumerate(test_cases, 1):
    print(f"📋 Test {i}: {test_case.get('topic', 'Default topic')}")
    result = app.invoke(test_case)
    print(f"📝 Result: {result.get('draft', 'No draft')[100:200]}...")
    print("─" * 50)

🚀 Testing article generation system...

📋 Test 1: LangChain multi-agent systems
Researching: LangChain multi-agent systems
📝 Result: icial intelligence, LangChain's multi-agent systems stand out as a game-changer for developers. By h...
──────────────────────────────────────────────────
📋 Test 2: Quantum Computing
Researching: Quantum Computing
📝 Result: ; it represents a paradigm shift in how we process information. By harnessing the principles of quan...
──────────────────────────────────────────────────
📋 Test 3: Default topic
Researching: AI technology
📝 Result:  (AI) is no longer just a buzzword; it’s a transformative force reshaping industries and redefining ...
──────────────────────────────────────────────────


## Advanced: Multi-Agent Debate

In [ ]:
def debate_agent1(state: dict):
    turn = state.get("turn", 0)
    response = assistant1.invoke(f"Round {turn}: Argue in favor of AI safety.")
    state[f"agent1_round_{turn}"] = response.content
    return state

def debate_agent2(state: dict):
    turn = state.get("turn", 0)
    msg = state[f"agent1_round_{turn}"]
    response = assistant2.invoke(f"Round {turn}: Rebuttal to: {msg}")
    state[f"agent2_round_{turn}"] = response.content
    state["turn"] = turn + 1
    return state

# Conditional edge: keep debating until turn = 3
def should_continue(state: dict):
    return "debate" if state["turn"] < 3 else END


In [19]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Define two LLMs with different perspectives
assistant1 = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)  # Pro-AI safety
assistant2 = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)  # Counter-arguments

def debate_agent1(state: dict):
    turn = state.get("turn", 0)
    print(f"🤖 Agent 1 - Round {turn}")
    
    response = assistant1.invoke(f"""Round {turn}: You strongly believe in AI safety regulations. 
Argue persuasively about why AI needs strict safety measures and oversight.
Keep your argument concise and compelling.""")
    
    state[f"agent1_round_{turn}"] = response.content
    state["last_speaker"] = "agent1"
    return state

def debate_agent2(state: dict):
    turn = state.get("turn", 0)
    print(f"🤖 Agent 2 - Round {turn}")
    
    # Get the previous argument
    previous_arg = state.get(f"agent1_round_{turn}", "No previous argument")
    
    response = assistant2.invoke(f"""Round {turn}: You are skeptical about excessive AI regulation.
Rebuttal to: {previous_arg}

Argue for innovation freedom while addressing safety concerns.
Keep your response concise and counter the previous points effectively.""")
    
    state[f"agent2_round_{turn}"] = response.content
    state["last_speaker"] = "agent2"
    state["turn"] = turn + 1  # Increment turn counter
    return state

# Conditional edge: keep debating until turn = 3
def should_continue(state: dict):
    current_turn = state.get("turn", 0)
    print(f"🔄 Checking if should continue: turn {current_turn}")
    return "debate" if current_turn < 3 else END

# Build the debate graph
workflow = StateGraph(dict)

# Add nodes
workflow.add_node("agent1", debate_agent1)
workflow.add_node("agent2", debate_agent2)

# Set entry point
workflow.set_entry_point("agent1")

# Add edges with conditional routing
workflow.add_edge("agent1", "agent2")
workflow.add_conditional_edges("agent2", should_continue, {
    "debate": "agent1",  # Continue debating
    END: END            # End the debate
})

# Compile the app
app = workflow.compile()

# Run the debate
print("🎭 Starting AI Safety Debate (3 rounds)")
print("=" * 50)

initial_state = {"turn": 0, "topic": "AI Safety Regulations"}
final_state = app.invoke(initial_state)

print("\n" + "=" * 50)
print("🏁 DEBATE COMPLETE!")
print("=" * 50)

# Display the debate transcript
for turn in range(3):
    print(f"\n🔵 ROUND {turn} - AGENT 1 (Pro-Safety):")
    print(final_state.get(f"agent1_round_{turn}", "No argument recorded"))
    
    print(f"\n🔴 ROUND {turn} - AGENT 2 (Pro-Innovation):")
    print(final_state.get(f"agent2_round_{turn}", "No rebuttal recorded"))
    print("-" * 50)

print(f"\n📊 Final turn count: {final_state.get('turn', 0)}")

🎭 Starting AI Safety Debate (3 rounds)
🤖 Agent 1 - Round 0
🤖 Agent 2 - Round 0
🔄 Checking if should continue: turn 1
🤖 Agent 1 - Round 1
🤖 Agent 2 - Round 1
🔄 Checking if should continue: turn 2
🤖 Agent 1 - Round 2
🤖 Agent 2 - Round 2
🔄 Checking if should continue: turn 3

🏁 DEBATE COMPLETE!

🔵 ROUND 0 - AGENT 1 (Pro-Safety):
AI safety regulations are essential to ensure the responsible development and deployment of artificial intelligence technologies. As AI systems become increasingly powerful and integrated into critical sectors—such as healthcare, finance, and transportation—the potential risks associated with their misuse or malfunction also escalate. 

First, AI systems can inadvertently perpetuate biases present in their training data, leading to discriminatory outcomes that can harm marginalized communities. Without stringent oversight, these biases can go unchecked, exacerbating societal inequalities.

Second, AI technologies, particularly those capable of autonomous decision-

## enhanced

In [20]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

assistant1 = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)
assistant2 = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)

def debate_agent1(state: dict):
    turn = state.get("turn", 0)
    print(f"🎤 Agent 1 speaking (Round {turn + 1})...")
    
    # Get previous rebuttal if exists
    previous_rebuttal = state.get(f"agent2_round_{turn-1}", "") if turn > 0 else ""
    
    prompt = f"""Round {turn + 1}: You advocate for strong AI safety measures.

Previous counter-argument: {previous_rebuttal}

Make a compelling case for why AI development needs strict oversight, regulations, and safety protocols.
Focus on existential risks, ethical concerns, and real-world examples."""
    
    response = assistant1.invoke(prompt)
    state[f"agent1_round_{turn}"] = response.content
    return state

def debate_agent2(state: dict):
    turn = state.get("turn", 0)
    print(f"🎤 Agent 2 speaking (Round {turn + 1})...")
    
    current_argument = state.get(f"agent1_round_{turn}", "No argument provided")
    
    prompt = f"""Round {turn + 1}: You advocate for innovation-friendly AI development.

Current argument against you: {current_argument}

Provide a thoughtful rebuttal emphasizing:
- The importance of innovation pace
- Risks of over-regulation
- Balancing safety with progress
- Economic and societal benefits of AI"""
    
    response = assistant2.invoke(prompt)
    state[f"agent2_round_{turn}"] = response.content
    state["turn"] = turn + 1
    return state

def should_continue(state: dict):
    current_turn = state.get("turn", 0)
    return "debate" if current_turn < 3 else END

# Build graph
workflow = StateGraph(dict)
workflow.add_node("pro_safety", debate_agent1)
workflow.add_node("pro_innovation", debate_agent2)

workflow.set_entry_point("pro_safety")
workflow.add_edge("pro_safety", "pro_innovation")
workflow.add_conditional_edges("pro_innovation", should_continue, {
    "debate": "pro_safety",
    END: END
})

app = workflow.compile()

# Run with different topics
debate_topics = [
    "AI Safety Regulations",
    "Universal Basic Income in AI age", 
    "Space Exploration Priority"
]

for topic in debate_topics:
    print(f"\n🎭 DEBATE TOPIC: {topic}")
    print("=" * 60)
    
    result = app.invoke({"turn": 0, "topic": topic})
    
    print(f"\n🏁 Debate completed in {result['turn']} rounds")
    print("=" * 60)


🎭 DEBATE TOPIC: AI Safety Regulations
🎤 Agent 1 speaking (Round 1)...
🎤 Agent 2 speaking (Round 1)...
🎤 Agent 1 speaking (Round 2)...
🎤 Agent 2 speaking (Round 2)...
🎤 Agent 1 speaking (Round 3)...
🎤 Agent 2 speaking (Round 3)...

🏁 Debate completed in 3 rounds

🎭 DEBATE TOPIC: Universal Basic Income in AI age
🎤 Agent 1 speaking (Round 1)...
🎤 Agent 2 speaking (Round 1)...
🎤 Agent 1 speaking (Round 2)...
🎤 Agent 2 speaking (Round 2)...
🎤 Agent 1 speaking (Round 3)...
🎤 Agent 2 speaking (Round 3)...

🏁 Debate completed in 3 rounds

🎭 DEBATE TOPIC: Space Exploration Priority
🎤 Agent 1 speaking (Round 1)...
🎤 Agent 2 speaking (Round 1)...
🎤 Agent 1 speaking (Round 2)...
🎤 Agent 2 speaking (Round 2)...
🎤 Agent 1 speaking (Round 3)...
🎤 Agent 2 speaking (Round 3)...

🏁 Debate completed in 3 rounds


In [25]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

# Use a deterministic model for consistency
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Define a proper state structure using TypedDict
class ChatState(TypedDict):
    user_input: str
    research: str
    draft: str
    feedback: str

# --- Agents ---
def researcher(state: ChatState):
    topic = state.get("user_input", "AI")
    response = llm.invoke(f"You are a researcher. Find 3 key facts about {topic}.")
    state["research"] = response.content
    print("\n[Researcher] 📚", response.content)
    return state

def writer(state: ChatState):
    research = state["research"]
    feedback = state.get("feedback", "")
    prompt = f"You are a writer. Draft a short article using this research:\n{research}\n"
    if feedback:
        prompt += f"\nRevise according to this feedback: {feedback}"
    response = llm.invoke(prompt)
    state["draft"] = response.content
    print("\n[Writer] ✍️", response.content)
    return state

def critic(state: ChatState):
    draft = state["draft"]
    response = llm.invoke(
        f"You are a critic. Review this draft:\n{draft}\n"
        f"Give constructive feedback. If the draft is good, just reply 'APPROVED'."
    )
    state["feedback"] = response.content
    print("\n[Critic] 🧐", response.content)
    return state

# --- Conditional loop ---
def should_continue(state: ChatState):
    feedback = state.get("feedback", "")
    if "APPROVED" in feedback.upper():
        return END
    return "writer"

# --- Graph Setup ---
workflow = StateGraph(ChatState)

workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)
workflow.add_node("critic", critic)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")
workflow.add_conditional_edges("critic", should_continue)

app = workflow.compile()

# --- Interactive Chat ---
def multi_agent_chat():
    print("🤖 Multi-Agent Chatroom: Researcher + Writer + Critic")
    print("Type a topic and watch the agents collaborate. Type 'quit' to exit.\n")

    while True:
        user_input = input("🟢 You: ")
        if user_input.lower() in ["quit", "exit"]:
            break

        # Initialize state with proper structure
        state: ChatState = {
            "user_input": user_input,
            "research": "",
            "draft": "",
            "feedback": ""
        }
        
        try:
            final_state = app.invoke(state)
            print("\n=== ✅ Final Approved Draft ===")
            print(final_state["draft"])
            print("==============================\n")
        except Exception as e:
            print(f"❌ Error: {e}")
            print("Please try again with a different topic.\n")

multi_agent_chat()

🤖 Multi-Agent Chatroom: Researcher + Writer + Critic
Type a topic and watch the agents collaborate. Type 'quit' to exit.




[Researcher] 📚 Certainly! Here are three key facts about artificial intelligence (AI):

1. **Types of AI**: AI can be categorized into three main types: Narrow AI, General AI, and Superintelligent AI. Narrow AI, which is the most common today, is designed to perform specific tasks (e.g., language translation, image recognition). General AI, which remains largely theoretical, would possess the ability to understand, learn, and apply intelligence across a wide range of tasks, similar to a human. Superintelligent AI refers to a hypothetical future AI that surpasses human intelligence in all aspects.

2. **Machine Learning and Deep Learning**: A significant subset of AI is machine learning (ML), which involves algorithms that allow computers to learn from and make predictions based on data. Deep learning, a further subset of ML, uses neural networks with many layers (hence "deep") to analyze various forms of data, such as images and text. These technologies have driven many recent advance

KeyboardInterrupt: 

In [23]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Use a deterministic model for consistency
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Agents ---
def researcher(state: dict):
    topic = state.get("user_input", "AI")
    print(f"\n[Researcher] 🔍 Researching: {topic}")
    
    response = llm.invoke(f"You are a researcher. Find 3 key facts about {topic}.")
    state["research"] = response.content
    print("[Researcher] 📚 Research completed")
    return state

def writer(state: dict):
    research = state.get("research", "")
    feedback = state.get("feedback", "")
    
    print(f"\n[Writer] ✍️ Writing draft...")
    prompt = f"You are a writer. Draft a short article using this research:\n{research}\n"
    if feedback:
        prompt += f"\nRevise according to this feedback: {feedback}"
    
    response = llm.invoke(prompt)
    state["draft"] = response.content
    print("[Writer] 📝 Draft completed")
    return state

def critic(state: dict):
    draft = state.get("draft", "")
    print(f"\n[Critic] 🧐 Reviewing draft...")
    
    response = llm.invoke(
        f"You are a critic. Review this draft:\n{draft}\n"
        f"Give constructive feedback. If the draft is good, just reply 'APPROVED'."
    )
    state["feedback"] = response.content
    print("[Critic] ✅ Feedback provided")
    return state

# --- Conditional loop ---
def should_continue(state: dict):
    feedback = state.get("feedback", "")
    print(f"\n[Decision] 🤔 Checking feedback: {feedback[:50]}...")
    
    if "APPROVED" in feedback.upper():
        print("[Decision] 🎉 Draft approved! Ending workflow.")
        return END
    else:
        print("[Decision] 🔄 Sending back to writer for revision.")
        return "writer"

# --- Graph Setup ---
workflow = StateGraph(dict)  # Use dict instead of custom class

workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)
workflow.add_node("critic", critic)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")
workflow.add_conditional_edges("critic", should_continue)

app = workflow.compile()

# --- Interactive Chat ---
def multi_agent_chat():
    print("🤖 Multi-Agent Chatroom: Researcher + Writer + Critic")
    print("Type a topic and watch the agents collaborate. Type 'quit' to exit.\n")

    while True:
        user_input = input("🟢 You: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            break

        print("\n" + "="*60)
        print(f"🚀 Processing: '{user_input}'")
        print("="*60)
        
        # Use regular dictionary instead of ChatState
        state = {"user_input": user_input}
        final_state = app.invoke(state)

        print("\n" + "="*60)
        print("✅ FINAL APPROVED DRAFT")
        print("="*60)
        print(final_state.get("draft", "No draft was generated"))
        print("="*60)
        
        # Show revision history if available
        if "feedback" in final_state and "APPROVED" not in final_state["feedback"].upper():
            print("\n📋 FEEDBACK RECEIVED:")
            print(final_state["feedback"])
            print("="*60)

multi_agent_chat()

🤖 Multi-Agent Chatroom: Researcher + Writer + Critic
Type a topic and watch the agents collaborate. Type 'quit' to exit.


🚀 Processing: 'AI'

[Researcher] 🔍 Researching: AI
[Researcher] 📚 Research completed

[Writer] ✍️ Writing draft...
[Writer] 📝 Draft completed

[Critic] 🧐 Reviewing draft...
[Critic] ✅ Feedback provided

[Decision] 🤔 Checking feedback: The draft is well-structured and provides a clear ...
[Decision] 🔄 Sending back to writer for revision.

[Writer] ✍️ Writing draft...
[Writer] 📝 Draft completed

[Critic] 🧐 Reviewing draft...
[Critic] ✅ Feedback provided

[Decision] 🤔 Checking feedback: The draft is well-structured and covers essential ...
[Decision] 🔄 Sending back to writer for revision.

[Writer] ✍️ Writing draft...
[Writer] 📝 Draft completed

[Critic] 🧐 Reviewing draft...
[Critic] ✅ Feedback provided

[Decision] 🤔 Checking feedback: The draft is well-structured and provides a clear ...
[Decision] 🔄 Sending back to writer for revision.

[Writer] ✍️ Writing draft...


KeyboardInterrupt: 

In [26]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import Dict, Any

# Use a deterministic model for consistency
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Define state as a regular dictionary
State = Dict[str, Any]

# --- Agents ---
def researcher(state: State):
    topic = state.get("user_input", "AI")
    response = llm.invoke(f"You are a researcher. Find 3 key facts about {topic}.")
    return {"research": response.content}

def writer(state: State):
    research = state["research"]
    feedback = state.get("feedback", "")
    prompt = f"You are a writer. Draft a short article using this research:\n{research}\n"
    if feedback:
        prompt += f"\nRevise according to this feedback: {feedback}"
    response = llm.invoke(prompt)
    return {"draft": response.content}

def critic(state: State):
    draft = state["draft"]
    response = llm.invoke(
        f"You are a critic. Review this draft:\n{draft}\n"
        f"Give constructive feedback. If the draft is good, just reply 'APPROVED'."
    )
    return {"feedback": response.content}

# --- Conditional loop ---
def should_continue(state: State):
    feedback = state.get("feedback", "")
    if "APPROVED" in feedback.upper():
        return END
    return "writer"

# --- Graph Setup ---
workflow = StateGraph(State)

workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)
workflow.add_node("critic", critic)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")
workflow.add_conditional_edges("critic", should_continue)

app = workflow.compile()

# --- Interactive Chat ---
def multi_agent_chat():
    print("🤖 Multi-Agent Chatroom: Researcher + Writer + Critic")
    print("Type a topic and watch the agents collaborate. Type 'quit' to exit.\n")

    while True:
        user_input = input("🟢 You: ")
        if user_input.lower() in ["quit", "exit"]:
            break

        # Initialize state
        state = {"user_input": user_input}
        
        try:
            final_state = app.invoke(state)
            print("\n[Researcher] 📚", final_state.get("research", "No research found"))
            print("\n[Writer] ✍️", final_state.get("draft", "No draft found"))
            print("\n[Critic] 🧐", final_state.get("feedback", "No feedback found"))
            
            print("\n=== ✅ Final Approved Draft ===")
            print(final_state.get("draft", "No final draft available"))
            print("==============================\n")
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()
            print("Please try again with a different topic.\n")

multi_agent_chat()

🤖 Multi-Agent Chatroom: Researcher + Writer + Critic
Type a topic and watch the agents collaborate. Type 'quit' to exit.


[Researcher] 📚 No research found

[Writer] ✍️ No draft found

[Critic] 🧐 APPROVED

=== ✅ Final Approved Draft ===
No final draft available


[Researcher] 📚 No research found

[Writer] ✍️ No draft found

[Critic] 🧐 APPROVED

=== ✅ Final Approved Draft ===
No final draft available



KeyboardInterrupt: Interrupted by user

### 🔹 Multi-Agent Chat with Memory (Researcher + Writer + Critic)

In [28]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Deterministic model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- State with memory ---
def init_state():
    return {"history": []}

# --- Agents ---
def researcher(state: dict):
    topic = state.get("user_input", "AI")
    response = llm.invoke(f"You are a researcher. Find 3 key facts about {topic}.")
    state["research"] = response.content
    state["history"].append({"role": "Researcher", "message": response.content})
    print("\n[Researcher] 📚", response.content)
    return state

def writer(state: dict):
    research = state["research"]
    feedback = state.get("feedback", "")
    prompt = f"You are a writer. Draft a short article using this research:\n{research}\n"
    if feedback:
        prompt += f"\nRevise according to this feedback: {feedback}"
    response = llm.invoke(prompt)
    state["draft"] = response.content
    state["history"].append({"role": "Writer", "message": response.content})
    print("\n[Writer] ✍️", response.content)
    return state

def critic(state: dict):
    draft = state["draft"]
    response = llm.invoke(
        f"You are a critic. Review this draft:\n{draft}\n"
        f"Give constructive feedback. If the draft is good, just reply 'APPROVED'."
    )
    state["feedback"] = response.content
    state["history"].append({"role": "Critic", "message": response.content})
    print("\n[Critic] 🧐", response.content)
    return state

# --- Conditional loop ---
def should_continue(state: dict):
    feedback = state.get("feedback", "")
    if "APPROVED" in feedback.upper():
        return END
    return "writer"

# --- Graph Setup ---
workflow = StateGraph(dict)

workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)
workflow.add_node("critic", critic)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")
workflow.add_conditional_edges("critic", should_continue)

app = workflow.compile()

# --- Interactive Chat with Memory ---
def multi_agent_chat_with_memory():
    print("🤖 Multi-Agent Chatroom with Memory (Researcher + Writer + Critic)")
    print("Type a topic and watch the agents collaborate. Type 'history' to see memory. Type 'quit' to exit.\n")

    state = init_state()

    while True:
        user_input = input("🟢 You: ")
        if user_input.lower() in ["quit", "exit"]:
            break
        if user_input.lower() == "history":
            print("\n=== Conversation History ===")
            for h in state["history"]:
                print(f"[{h['role']}] {h['message']}\n")
            continue

        state["user_input"] = user_input
        final_state = app.invoke(state)

        print("\n=== ✅ Final Approved Draft ===")
        print(final_state["draft"])
        print("==============================\n")

if __name__ == "__main__":
    multi_agent_chat_with_memory()


🤖 Multi-Agent Chatroom with Memory (Researcher + Writer + Critic)
Type a topic and watch the agents collaborate. Type 'history' to see memory. Type 'quit' to exit.




[Researcher] 📚 Certainly! Here are three key facts about the impact of artificial intelligence (AI) on jobs:

1. **Job Displacement vs. Job Creation**: While AI and automation can lead to the displacement of certain jobs, particularly those involving repetitive tasks, they also create new job opportunities in emerging fields. For instance, roles in AI development, data analysis, and machine learning are on the rise. The World Economic Forum has projected that while 85 million jobs may be displaced by 2025 due to the shift in labor between humans and machines, 97 million new roles may emerge that are more adapted to the new division of labor.

2. **Skill Shift and Reskilling Needs**: The integration of AI into the workforce necessitates a shift in skills. Many jobs will require a higher level of technical proficiency, critical thinking, and emotional intelligence. As a result, there is an increasing demand for reskilling and upskilling programs to help workers transition into roles tha

KeyboardInterrupt: 

### Multi-Agent Group Chat (LangGraph)

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Deterministic model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Shared State ---
class ChatState(dict):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.setdefault("history", [])
        self.setdefault("turn", 0)

    def add_to_history(self, role, message):
        self["history"].append({"role": role, "message": message})

    def last_messages(self, n=5):
        return "\n".join([f"{h['role']}: {h['message']}" for h in self["history"][-n:]])


# --- Agent Functions ---
def researcher(state: ChatState):
    context = state.last_messages()
    response = llm.invoke(
        f"You are Researcher. Read the chat so far and contribute facts or insights.\n"
        f"Chat so far:\n{context}\n"
        f"Your turn:"
    )
    state.add_to_history("Researcher", response.content)
    print("\n[Researcher] 📚", response.content)
    return state

def writer(state: ChatState):
    context = state.last_messages()
    response = llm.invoke(
        f"You are Writer. Read the chat so far and improve or extend the draft/article.\n"
        f"Chat so far:\n{context}\n"
        f"Your turn:"
    )
    state.add_to_history("Writer", response.content)
    print("\n[Writer] ✍️", response.content)
    return state

def critic(state: ChatState):
    context = state.last_messages()
    response = llm.invoke(
        f"You are Critic. Read the chat so far and give constructive feedback.\n"
        f"Chat so far:\n{context}\n"
        f"Your turn:"
    )
    state.add_to_history("Critic", response.content)
    print("\n[Critic] 🧐", response.content)
    return state

def user_node(state: ChatState):
    user_input = state.get("user_input", "")
    if user_input:
        state.add_to_history("User", user_input)
        print("\n[🟢 You]", user_input)
    return state


# --- Graph Setup ---
workflow = StateGraph(ChatState)

workflow.add_node("user", user_node)
workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)
workflow.add_node("critic", critic)

workflow.set_entry_point("user")

# Sequential cycle: User → Researcher → Writer → Critic → back to User
workflow.add_edge("user", "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")
workflow.add_edge("critic", "user")

app = workflow.compile()


# --- Interactive Group Chat ---
def multi_agent_group_chat():
    print("🤖 Multi-Agent Group Chatroom (Researcher + Writer + Critic)")
    print("Type a message to join the conversation. Type 'history' to view log. Type 'quit' to exit.\n")

    state = ChatState()

    while True:
        user_input = input("🟢 You: ")
        if user_input.lower() in ["quit", "exit"]:
            break
        if user_input.lower() == "history":
            print("\n=== Conversation History ===")
            for h in state["history"]:
                print(f"[{h['role']}] {h['message']}\n")
            continue

        # Feed user message and run one round
        state["user_input"] = user_input
        state = app.invoke(state)


multi_agent_group_chat()


In [33]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Deterministic model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Helper functions for state ---
def make_state():
    return {"history": [], "turn": 0, "user_input": ""}

def add_to_history(state, role, message):
    state["history"].append({"role": role, "message": message})

def last_messages(state, n=5):
    return "\n".join([f"{h['role']}: {h['message']}" for h in state["history"][-n:]])

# --- Agent Functions ---
def researcher(state):
    context = last_messages(state)
    response = llm.invoke(
        f"You are Researcher. Read the chat so far and contribute facts or insights.\n"
        f"Chat so far:\n{context}\nYour turn:"
    )
    add_to_history(state, "Researcher", response.content)
    print("\n[Researcher] 📚", response.content)
    return state

def writer(state):
    context = last_messages(state)
    response = llm.invoke(
        f"You are Writer. Read the chat so far and improve or extend the draft/article.\n"
        f"Chat so far:\n{context}\nYour turn:"
    )
    add_to_history(state, "Writer", response.content)
    print("\n[Writer] ✍️", response.content)
    return state

def critic(state):
    context = last_messages(state)
    response = llm.invoke(
        f"You are Critic. Read the chat so far and give constructive feedback.\n"
        f"Chat so far:\n{context}\nYour turn:"
    )
    add_to_history(state, "Critic", response.content)
    print("\n[Critic] 🧐", response.content)
    return state

def user_node(state):
    user_input = state.get("user_input", "")
    if user_input:
        add_to_history(state, "User", user_input)
        print("\n[🟢 You]", user_input)
    return state

# --- Graph Setup ---
workflow = StateGraph(dict)  # Just pass dict; we handle helpers

workflow.add_node("user", user_node)
workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)
workflow.add_node("critic", critic)

workflow.set_entry_point("user")

# Sequential cycle
workflow.add_edge("user", "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")
workflow.add_edge("critic", "user")

app = workflow.compile()

# --- Jupyter-Friendly Runner ---
state = make_state()

def chat_round(user_message: str):
    """
    Run one full cycle: User → Researcher → Writer → Critic.
    """
    global state
    state["user_input"] = user_message
    state = app.invoke(state)
    return state

def show_history():
    """
    Pretty-print the conversation history.
    """
    for h in state["history"]:
        print(f"[{h['role']}] {h['message']}\n")


In [ ]:
chat_round("Let's brainstorm article ideas about AI and education.")
# show_history()


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Jupyter Chat UI ---
output = widgets.Output()

input_box = widgets.Text(
    placeholder="Type your message here...",
    description="You:",
    layout=widgets.Layout(width="80%")
)

send_button = widgets.Button(
    description="Send",
    button_style="success"
)

# --- Callback ---
def on_send_clicked(b):
    with output:
        user_message = input_box.value
        if not user_message.strip():
            return
        input_box.value = ""
        
        # Run one chat round
        chat_round(user_message)
        
        # Clear previous output and show full history
        clear_output(wait=True)
        show_history()

send_button.on_click(on_send_clicked)

# --- Display UI ---
display(input_box, send_button, output)
